# Simple: Uniform Temporal Noise with PAMOLA.CORE

**Goal**: Learn temporal noise basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample datetime data from examples/data_examples/
- Apply uniform random time shifts using execute()
- Compare before/after temporal distributions
- Understand privacy-utility tradeoffs for datetime anonymization

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/
        └── 01_uniform_temporal_noise_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime, timedelta
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.noise.uniform_temporal_op import UniformTemporalNoiseOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from `examples/data_examples/sample.csv`
- Auto-creates sample data if file missing
- Review dataset structure focusing on datetime fields

**What you'll see:**
- Dataset summary (100 records, 4 columns)
- First 5 records with patient visits and timestamps
- Column statistics (types, date ranges, unique counts)
- Datetime field auto-converted to pandas datetime format

**Note:** Auto-generated sample includes 100 patient visits across 2024 (8 AM-6 PM business hours) for realistic temporal patterns

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create realistic patient visit data
    np.random.seed(42)
    base_date = pd.Timestamp('2024-01-01')
    
    sample_data = pd.DataFrame({
        'patient_id': [f'P{i:04d}' for i in range(1, 101)],
        'visit_datetime': [base_date + timedelta(days=np.random.randint(0, 365), 
                                                  hours=np.random.randint(8, 18),
                                                  minutes=np.random.randint(0, 60)) 
                           for _ in range(100)],
        'department': np.random.choice(['Cardiology', 'Neurology', 'Orthopedics', 'Pediatrics'], 100),
        'visit_type': np.random.choice(['Consultation', 'Follow-up', 'Emergency', 'Routine'], 100)
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
if 'visit_datetime' in df.columns:
    df['visit_datetime'] = pd.to_datetime(df['visit_datetime'])

print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:20s}): {df[col].nunique()} unique values")
    elif pd.api.types.is_datetime64_any_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:20s}): {df[col].min()} to {df[col].max()}")
    else:
        print(f"  {col:20s} ({dtype_str:20s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE field_name**: Edit `create_config_kwargs()` function
   - Default: `"snapshot_date"`
   - Change to YOUR datetime field (e.g., `"visit_datetime"`)
2. Run to validate field and setup environment

**What you'll see:**
- Field validation (type, date range, time span in days)
- Auto-conversion to pandas datetime format
- Task directory, TaskReporter, DataSource initialized (✓)

**Note:** Field must contain parseable datetime values. Invalid dates will cause errors

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "snapshot_date"
    }

kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

# Convert field to datetime series
df[field_name] = pd.to_datetime(df[field_name], errors='coerce')

print(f"✓ Field to process: '{field_name}'")
print(f"  Data type: {df[field_name].dtype}")
print(f"  Non-null values: {df[field_name].notna().sum()}")

min_val = df[field_name].min()
max_val = df[field_name].max()

print(f"  Date range: {min_val} to {max_val}")

if pd.notna(min_val) and pd.notna(max_val):
    print(f"  Time span: {(max_val - min_val).days} days")
else:
    print("  Time span: N/A (invalid or non-date values)")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_temporal_001",
    task_type="temporal_anonymization",
    description="Uniform temporal noise for patient visit timestamps",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration summary
- Run to execute temporal noise addition
- Monitor progress bar (6 steps: validate → generate → transform → constrain → round → finalize)

**Key parameters:**
- `noise_range_days=7`: Random shift of ±7 days
- `direction='both'`: Allow forward or backward shifts
- `output_granularity='hour'`: Round to nearest hour
- `preserve_weekends=False`: Don't maintain weekday status
- `use_secure_random=True`: Cryptographic randomization

**What you'll see:**
- Configuration summary with all parameters
- Progress bar through 6 processing steps
- Completion status with artifact count (4-6 files)
- Success message (✅ Operation completed!)

**Note:** Execution takes ~2-5 seconds for 100 records

In [ ]:
# Create operation with production-style configuration
operation = UniformTemporalNoiseOperation(
    # Core parameters
    field_name=field_name,
    mode='REPLACE',
    
    # Output configuration
    column_prefix='_',
    output_field_name=None,
    output_format='csv',
    
    # Temporal noise parameters
    noise_range_days=7,           # ±7 days random shift
    noise_range_hours=None,       # Additional hours shift (optional)
    noise_range_minutes=None,     # Additional minutes shift (optional)
    noise_range_seconds=None,     # Additional seconds shift (optional)
    
    # Direction control
    direction='both',             # 'both', 'forward', or 'backward'
    
    # Boundary constraints
    min_datetime=None,            # Minimum allowed datetime
    max_datetime=None,            # Maximum allowed datetime
    
    # Special date handling
    preserve_special_dates=False, # Preserve specific dates unchanged
    special_dates=None,           # List of dates to preserve
    preserve_weekends=False,      # Maintain weekend/weekday status
    preserve_time_of_day=False,   # Only shift date, keep time
    
    # Granularity control
    output_granularity='hour',    # 'day', 'hour', 'minute', 'second', or None
    
    # Randomness control
    random_seed=None,             # For reproducibility (ignored if secure=True)
    use_secure_random=True,       # Cryptographically secure random
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Field:              {operation.field_name}")
print(f"  Noise range:        ±{operation.noise_range_days} days")
print(f"  Direction:          {operation.direction}")
print(f"  Output granularity: {operation.output_granularity}")
print(f"  Secure random:      {operation.use_secure_random}")
print(f"  Preserve weekends:  {operation.preserve_weekends}")
print(f"  Preserve time:      {operation.preserve_time_of_day}")
print(f"  Visualizations:     {operation.generate_visualization}")
print(f"  Save output:        {operation.save_output}")
print(f"  Force recalc:       {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing Uniform Temporal Noise Operation...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load and compare shifted timestamps
- Review temporal shift statistics

**What you'll see:**
- Artifacts list (CSV, JSON, PNG files)
- Sample records (first 10) with shifted timestamps
- Temporal shift analysis (mean, std, min, max shifts in days)
- Shift direction counts (forward, backward, zero)
- Date range comparison (original vs anonymized)

**Note:** Mean shift ≈ 0 days indicates balanced forward/backward shifts. Zero shifts occur due to granularity rounding or boundary constraints

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    result_df[field_name] = pd.to_datetime(result_df[field_name])
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records
    print("\n🔍 Sample Processed Records:")
    display_cols = [field_name] + [col for col in df.columns if col != field_name]
    print(result_df[display_cols].head(10))
    
    # Calculate time shifts
    time_shifts = (result_df[field_name] - df[field_name]).dt.total_seconds() / 86400  # Convert to days
    
    print("\n📈 Temporal Shift Analysis:")
    print(f"  Mean shift:         {time_shifts.mean():.2f} days")
    print(f"  Std deviation:      {time_shifts.std():.2f} days")
    print(f"  Min shift:          {time_shifts.min():.2f} days")
    print(f"  Max shift:          {time_shifts.max():.2f} days")
    print(f"  Forward shifts:     {(time_shifts > 0).sum()} records")
    print(f"  Backward shifts:    {(time_shifts < 0).sum()} records")
    print(f"  Zero shifts:        {(time_shifts == 0).sum()} records")
    
    # Date range comparison
    print("\n📅 Date Range Comparison:")
    print(f"  Original range:     {df[field_name].min()} to {df[field_name].max()}")
    print(f"  Anonymized range:   {result_df[field_name].min()} to {result_df[field_name].max()}")
    print(f"  Original span:      {(df[field_name].max() - df[field_name].min()).days} days")
    print(f"  Anonymized span:    {(result_df[field_name].max() - result_df[field_name].min()).days} days")
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original records:   {len(df)}")
    print(f"  Processed records:  {len(result_df)}")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:15]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:35s}: {value:.4f}")
                else:
                    print(f"  {key:35s}: {value}")
            elif isinstance(value, dict) and key in ['noise_range_config', 'actual_shifts', 'shift_direction']:
                print(f"  {key}:")
                for subkey, subval in value.items():
                    if isinstance(subval, (int, float)):
                        if isinstance(subval, float):
                            print(f"    {subkey:30s}: {subval:.4f}")
                        else:
                            print(f"    {subkey:30s}: {subval}")
    
    print(f"\n💡 Temporal Noise Insight:")
    print(f"   Timestamps have been randomly shifted by up to ±{operation.noise_range_days} days,")
    print(f"   then rounded to {operation.output_granularity}-level precision for privacy protection.")
    print(f"   This preserves temporal patterns while preventing exact timestamp linkage.")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, metrics/, visualizations/, config/)
- File count per directory (typically 1-3 files each)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Time-shifted CSV
├── metrics/          # Shift statistics JSON
├── visualizations/   # Temporal distribution charts
└── config/           # Operation configuration
```

**Note:** Copy absolute path to file explorer for manual browsing

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Auto-loads NEWEST files from each category
- Focus on validation points in output

**What you'll see:**
1. **Metrics JSON**: Noise config, actual shifts (mean/std/min/max), direction breakdown, preservation settings
2. **Output Data**: First 10 records, date range, monthly distribution
3. **Visualizations**: Temporal distribution and shift pattern charts

**Validate:**
- ✅ Mean shift ≈ 0 for 'both' direction (balanced)
- ✅ Shifts span full configured range (privacy)
- ✅ Temporal patterns preserved (utility)
- ✅ Direction counts match configuration

**Note:** Only newest files shown. Multiple runs create versions - older artifacts excluded automatically

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST) 
metrics_files = list((task_dir / 'metrics').glob('*.json'))
if metrics_files:
    # 1) Exclude data_types inside IF
    filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

    if filtered:
        # Use only filtered files
        target_files = filtered
        print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
    else:
        # Fallback: nothing left after filtering → use all
        target_files = metrics_files
        print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

    # 2) Pick latest
    target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    latest_metrics_file = target_files[0]

    print(f"📄 Loading LATEST: {latest_metrics_file.name}")
    mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
    print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
    print("=" * 80)
    
    with open(latest_metrics_file) as f:
        data = json.load(f)
    
    # Display key sections
    if 'noise_range_config' in data:
        print("\n🔧 Noise Configuration:")
        for k, v in data['noise_range_config'].items():
            print(f"  {k:20s}: {v}")
    
    if 'actual_shifts' in data:
        print("\n📊 Actual Shift Statistics:")
        for k, v in data['actual_shifts'].items():
            if isinstance(v, float):
                print(f"  {k:20s}: {v:.4f}")
            else:
                print(f"  {k:20s}: {v}")
    
    if 'shift_direction' in data:
        print("\n↔️  Shift Direction Breakdown:")
        for k, v in data['shift_direction'].items():
            print(f"  {k:20s}: {v}")
    
    if 'preservation' in data:
        print("\n🛡️  Preservation Settings:")
        for k, v in data['preservation'].items():
            print(f"  {k:30s}: {v}")
else:
    print("⚠️  No metrics files found.")

# 2. OUTPUT DATA (NEWEST)
output_files = list((task_dir / 'output').glob('*.csv'))
if output_files:
    output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    latest = output_files[0]
    output_df = pd.read_csv(latest)
    output_df[field_name] = pd.to_datetime(output_df[field_name])
    
    print("\n\n📊 Output Data Sample:")
    print("=" * 80)
    print(output_df.head(10))
    
    # Show temporal distribution
    print("\n📅 Temporal Distribution:")
    print(f"  Date range: {output_df[field_name].min()} to {output_df[field_name].max()}")
    print(f"  Total span: {(output_df[field_name].max() - output_df[field_name].min()).days} days")
    
    # Monthly distribution
    monthly = output_df[field_name].dt.to_period('M').value_counts().sort_index()
    print(f"\n  Records by month:")
    for month, count in monthly.head(5).items():
        print(f"    {month}: {count} visits")
else:
    print("⚠️  No output data files found.")

# 3. VISUALIZATIONS (NEWEST BATCH)
viz_files = list((task_dir / 'visualizations').glob('*.png'))
if viz_files:
    viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    latest_time = viz_files[0].stat().st_mtime
    latest_batch = [f for f in viz_files if abs(f.stat().st_mtime - latest_time) < 10]
    
    print("\n\n🎨 VISUALIZATIONS")
    print("=" * 80)
    
    for viz_file in latest_batch:
        print(f"\n{viz_file.stem.replace('_', ' ').title()}")
        print("-" * 80)
        display(Image(filename=str(viz_file)))
else:
    print("⚠️  No visualization files found.")

print("\n\n✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ **Load data**: Load datetime data from examples/data_examples/  
✅ **Setup environment**: Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Configure uniform temporal noise with full parameters  
✅ **Execute**: Execute using operation.execute()  
✅ **Analyze results**: Analyze temporal shift patterns and review artifacts  

**Key takeaways:**
- Uniform temporal noise adds random time shifts uniformly distributed across a configured range
- Granularity control (hour/day/minute) balances privacy and utility
- Direction control (both/forward/backward) provides flexibility for different use cases
- Preservation options maintain important temporal patterns (weekends, special dates, time-of-day)
- Secure randomness ensures cryptographic-quality unpredictability

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_uniform_temporal_noise_advanced.ipynb`** includes:
- Boundary constraints (min/max datetime enforcement)
- Special date preservation for holidays/events
- Weekend preservation to maintain business patterns
- Time-of-day preservation for daily cycle analysis
- Multi-field temporal noise coordination
- Custom granularity patterns
- Reproducible vs. secure randomness comparison

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 📊 [Temporal Noise Best Practices](../../docs/guides/temporal_anonymization.md)